# 06 — Nhiệm vụ 4 (đối chứng): CLIP gốc + LoRA

Chạy CLIP gốc (không phải MedCLIP) + LoRA trên **cùng** BTXRD, cùng split, cùng số shot, cùng siêu tham số LoRA (`LORA_R`/`LORA_ALPHA`/`LORA_DROPOUT` trong `configs/config.py` — dùng chung với nhiệm vụ 3 để so sánh có kiểm soát). Chỉ đổi đúng 1 biến: backbone.

**Câu hỏi trả lời**: MedCLIP (pretrain trên X-quang *ngực*) có thật sự tốt hơn CLIP gốc (pretrain ảnh tổng quát) trên ảnh X-quang *xương*, hay domain y khoa không giúp ích gì nếu domain con (xương) khác domain pretrain (ngực)?

Cơ chế LoRA cho CLIP dùng `PlainMultiheadAttentionLoRA` (vendor từ CLIP-LoRA gốc, xử lý `nn.MultiheadAttention`) — khác `LinearLoRA` trực tiếp dùng cho MedCLIP (Q/K/V tách rời).

Yêu cầu: đã chạy `01_prepare_split.ipynb` trước đó. Nên chạy trên GPU.

In [ ]:
# Cell cài đặt — chạy trên Google Colab.
# Bỏ qua nếu chạy local (đã cài sẵn requirements + có sys.path đúng).

# 1) Mount Google Drive (nếu dataset/checkpoint lưu trên Drive)
# from google.colab import drive
# drive.mount('/content/drive')

# 2) Clone / trỏ tới thư mục project (sửa lại đường dẫn cho đúng chỗ bạn để code)
PROJECT_ROOT = '/content/drive/MyDrive/Code'  # <-- sửa lại nếu khác

# 3) Cài MedCLIP (editable) + dependency của src/ (bao gồm ftfy/regex cho package clip cục bộ)
# !pip install -e {PROJECT_ROOT}/MedCLIP --no-deps -q
# !pip install -r {PROJECT_ROOT}/src/requirements.txt -q

# 4) Thêm src/ vào sys.path để import được các module (configs, data, models, pipelines...)
import sys
sys.path.insert(0, f'{PROJECT_ROOT}/src')


### Bước 1 — CLIP zero-shot (mốc 0-shot riêng cho đường cong CLIP, không dùng chung mốc với MedCLIP)

In [ ]:
from pipelines.clip_zeroshot import run_clip_zeroshot

zeroshot_metrics = run_clip_zeroshot()
zeroshot_metrics


### Bước 2 — Sweep CLIP + LoRA qua nhiều mức shot × nhiều seed

In [ ]:
from pipelines.clip_lora_train import run_clip_lora_shots_sweep

# Mặc định dùng chung FEWSHOT_SWEEP_SHOTS/FEWSHOT_SWEEP_SEEDS và LORA_R/ALPHA/DROPOUT
# với nhiệm vụ 3 (MedCLIP) trong configs/config.py — để so sánh có kiểm soát.
results = run_clip_lora_shots_sweep()
results


### Bước 3 — So sánh trực tiếp MedCLIP + LoRA vs CLIP gốc + LoRA

In [ ]:
from compare_results import load_shots_curve, plot_backbone_comparison_curve, print_shots_curve_table, load_task_metrics
from configs import config as cfg
import os

medclip_curve = load_shots_curve(prefix='lora_fewshot', zeroshot_task='zeroshot')
clip_curve = load_shots_curve(prefix='clip_lora_fewshot', zeroshot_task='clip_zeroshot')

print('MedCLIP + LoRA:')
print_shots_curve_table(medclip_curve)
print('\nCLIP gốc + LoRA:')
print_shots_curve_table(clip_curve)

plot_backbone_comparison_curve(
    medclip_curve, clip_curve,
    baseline_metrics=load_task_metrics('baseline'),
    save_path=os.path.join(cfg.RESULTS_DIR, 'backbone_comparison_curve.png'),
)


In [ ]:
from IPython.display import Image

Image(os.path.join(cfg.RESULTS_DIR, 'backbone_comparison_curve.png'))
